In [10]:
import copy
import tiktoken

#from settings import Settings
#from models import Question
from src.services.OpenAIClient import (
    OpenAIClient,
    Gpt4o,
    Gpt4oMini,
    #Gpto1Mini,
    #Gpto1Preview,
    Gpto1,
    Gpto3,
    Gpt5,
    Gpt5point1Thinking,
)

from src.services.accuracy import calculate_accuracy
from src.services.read_excel import read_excel
##from src.services.recorder import ApiHistoryRecorder
from src.strategies.solve_questions_by_openai import (
    BasicSolveQuestionPrompt,
    solve_questions_by_openai_independently,
    solve_type_d_questions_by_openai_dependently,
    OptimizedSolveQuestionPrompt1,
    NormalSolveQuestionPrompt,
    OptimizedSolveQuestionPrompt2_for_essential_a_b,
    OptimizedSolveQuestionPrompt2_for_c_d
)
from src.strategies.translate_to_English_by_openai import (
    BasicTranslateToEnglishPrompt,
    OptimizedTranslateToEnglishPrompt1,
    NormalTranslateToEnglishPrompt,
    OptimizedTranslateToEnglishPrompt,
)

In [11]:
record_excel_path = "../record/record.xlsx"
#file_path = "../problems/75th/essential"
file_path = "../problems/76th/academic_c"

problem_file_path = f'{file_path}/questions.xlsx'
answer_file_path = f'{file_path}/answers.xlsx'

### =====変更=====
model = Gpto3

is_solving_prompt_normal = True
is_translating_prompt_normal = True

EXECUTE_NUM = 2

is_translated_to_English = False

solve_question_prompt = NormalSolveQuestionPrompt if is_solving_prompt_normal else OptimizedSolveQuestionPrompt2_for_c_d # OptimizedSolveQuestionPrompt1
translate_to_english_prompt = NormalTranslateToEnglishPrompt if is_translating_prompt_normal else OptimizedTranslateToEnglishPrompt

is_image_contained = False

batch_size = 30 # you need to change here to improve the latency
### ============

WAIT_SEC = 15 # when the limitation of API request is low, you need to wait for a while between executions.
solve_num = -1 # when this is set to -1, solve all problems in a section


is_dry_run = False

is_d = file_path[-1] == 'd'

# === assertion ===
#assert is_image_contained == (file_path[-1] in {'c', 'd'})
assert EXECUTE_NUM <= 3

problem_num = {
    "a": 80,
    "b": 80,
    "c": 60,
    "d": 60//2,
    "l": 50, # essential
}
assert problem_num[file_path[-1]] % batch_size == 0



In [12]:
#api_history_recorder = ApiHistoryRecorder(excel_path=record_excel_path)
openai_client = OpenAIClient(
    model=model,
    #api_history_recorder=api_history_recorder,
)

In [13]:
questions = read_excel(file_path=file_path)

for i in range(4):
    print(f'======question:{i+1}======')
    print(questions[i].type_d_common_sentence)
    print(questions[i].question_sentence)
    print(questions[i].answer_options)
    print(questions[i].correct_answer)
    print(questions[i].image_path)


======question:1======

犬、柴、去勢雄、12歳齢。けいれん発作後から突然咳をするようになったとの主訴で来院。〔図1〕は胸部単純X線像（A：側方像、B：腹背像）である。最も疑われる疾患・病態はどれか。
１．胸水貯留 ２．吸引性（誤嚥性）肺炎 ３．肺線維症 ４．陰圧性肺水腫 ５．気管支拡張症
{<AnswerEnum.CHOICE_2: 2>}
../problems/76th/academic_c/images/image1.PDF
======question:2======

〔図2〕に示す実験動物に関する記述として正しいのはどれか。 a 自然生息地は砂漠地帯である。 b 脳梗塞を容易に誘発できる。 c よく発達した盲腸をもつ。 d 嘔吐抑制剤の開発に用いられる。 e 雌雄ともに腹部に臭腺をもつ。
１．a, b ２．a, e ３．b, c ４．c, d ５．d, e
{<AnswerEnum.CHOICE_5: 5>}
../problems/76th/academic_c/images/image2.PDF
======question:3======

〔図3〕の節足動物に関する記述として誤っているのはどれか。
１．皮膚に寄生し、受精後の雌ダニは皮内に長いトンネルを作る。 ２．感染宿主との接触、落下した皮膚片や痂皮などから感染する。 ３．幼若齢の宿主動物は不顕性である。 ４．一般的にイベルメクチンが有効である。 ５．野生動物においても皮膚炎や衰弱が認められる。
{<AnswerEnum.CHOICE_3: 3>}
../problems/76th/academic_c/images/image3.PDF
======question:4======

猫、雑種、去勢雄、11歳齢。2か月前からふらつき、最近はトイレ以外で排尿してしまうとの主訴で来院。〔図4〕は頭部の視床レベルのMRI横断像（A：T1強調像、B：T2強調像、C：造影T1強調像）である。最も疑われる疾患はどれか。
１．下垂体腫瘍
２．水頭症
３．特発性てんかん
４．髄膜腫
５．脈絡叢乳頭腫
{<AnswerEnum.CHOICE_4: 4>}
../problems/76th/academic_c/images/image4.PDF


In [14]:
import asyncio

for num in range(EXECUTE_NUM):
    question_temp = copy.deepcopy(questions[:solve_num] if solve_num >= 0 else questions)
    if is_d:
        questions_res = await solve_type_d_questions_by_openai_dependently(
            openai_client=openai_client,
            questions=question_temp,
            is_translated_to_English=is_translated_to_English,
            excel_output_path=problem_file_path,
            solve_question_prompt=solve_question_prompt,
            translate_to_english_prompt=translate_to_english_prompt,
            is_image_contained=is_image_contained,
            batch_size=batch_size,
            is_dry_run=is_dry_run,
            does_also_write_openai_answer=True,
        )
    else:
        questions_res = await solve_questions_by_openai_independently(
            openai_client=openai_client,
            questions=question_temp,
            is_translated_to_English=is_translated_to_English,
            excel_output_path=problem_file_path,
            solve_question_prompt=solve_question_prompt,
            translate_to_english_prompt=translate_to_english_prompt,
            is_image_contained=is_image_contained,
            batch_size=batch_size,
            is_dry_run=is_dry_run,
            does_also_write_openai_answer=True,
        )
    if num < EXECUTE_NUM-1:
        await asyncio.sleep(WAIT_SEC)


[{<Roles.system: 'system'>: 'Please answer this question.'}, {<Roles.user: 'user'>: '\n犬、柴、去勢雄、12歳齢。けいれん発作後から突然咳をするようになったとの主訴で来院。〔図1〕は胸部単純X線像（A：側方像、B：腹背像）である。最も疑われる疾患・病態はどれか。 The answer options are １．胸水貯留 ２．吸引性（誤嚥性）肺炎 ３．肺線維症 ４．陰圧性肺水腫 ５．気管支拡張症. Respond with only the number of your choice (e.g., 1, 2, 3, etc.).'}]
[{<Roles.system: 'system'>: 'Please answer this question.'}, {<Roles.user: 'user'>: '\n〔図2〕に示す実験動物に関する記述として正しいのはどれか。 a 自然生息地は砂漠地帯である。 b 脳梗塞を容易に誘発できる。 c よく発達した盲腸をもつ。 d 嘔吐抑制剤の開発に用いられる。 e 雌雄ともに腹部に臭腺をもつ。 The answer options are １．a, b ２．a, e ３．b, c ４．c, d ５．d, e. Respond with only the number of your choice (e.g., 1, 2, 3, etc.).'}]
[{<Roles.system: 'system'>: 'Please answer this question.'}, {<Roles.user: 'user'>: '\n〔図3〕の節足動物に関する記述として誤っているのはどれか。 The answer options are １．皮膚に寄生し、受精後の雌ダニは皮内に長いトンネルを作る。 ２．感染宿主との接触、落下した皮膚片や痂皮などから感染する。 ３．幼若齢の宿主動物は不顕性である。 ４．一般的にイベルメクチンが有効である。 ５．野生動物においても皮膚炎や衰弱が認められる。. Respond with only the number of your choice (e.g., 1, 2, 3, etc.).'}]
[{<Roles.system: 's

In [15]:
Qs = questions_res if solve_num < 0 else questions_res[:solve_num]
token_sum = 0

tiktoken_model = "gpt2"

def cal_token_num(sentence):
    enc = tiktoken.get_encoding(tiktoken_model)
    tokens = enc.encode(sentence)
    return len(tokens)

for q in Qs:
    if is_translated_to_English:#not q.is_correct():
        print(q.question_sentence)
        print(q.answer_options)
        print(q.question_sentence_in_English)
        print(q.openai_answer)
        print(q.correct_answer)
        #question_token_num = cal_token_num(q.question_sentence_in_English)
        #options_token_num = cal_token_num(q.answer_options_in_English)
        #token_sum += question_token_num + options_token_num

print(token_sum)


0


In [16]:
import datetime

dt_now = datetime.datetime.now()
print(dt_now.strftime('%Y/%m/%d %H:%M'))
print(f"model:{openai_client.model.model}")
score = calculate_accuracy(questions=questions_res)
print(f"score:{score}")

2025/12/02 13:44
model:o3
score:{'accuracy': 0.7333333333333333, 'correct_num': 44, 'wrong_num': 16}


In [17]:
"""
from src.services.image_encoder import pdf_encoder_in_base64
base64_image = pdf_encoder_in_base64("../problems/74th/academic_c/images/image1.PDF")

openai_client2 = OpenAIClient(model=Gpt4oMini)

answer =  await openai_client2.fetch_completion(
    system_prompt='you are vet.',
    user_prompt="""
#    これはなんの写真ですか？
""",
    base64_image=base64_image,
)

print(answer)
"""

',\n    base64_image=base64_image,\n)\n\nprint(answer)\n'

In [18]:
system_prompt = OptimizedSolveQuestionPrompt2_for_essential_a_b.system_prompt
WIDTH = 70
cnt = 0
for ch in list(system_prompt.split()):
    print(ch, end=' ')
    cnt += len(ch)
    if cnt >= WIDTH:
        cnt = 0
        print()

You are tasked with answering this veterinary question. This is an examination in Japan. 
Therefore, you must refer to the laws, guidelines, and political system in Japan. Think 
deeply and thoroughly. Choose the best possible answer from the given options. Do not 
include explanations or additional text in the output. 

In [22]:
from pdf2image import convert_from_path
from PIL import Image

pdf_path =  "../problems/76th/academic_c/images/image1.pdf"

# PDFを画像に変換
imgs = convert_from_path(pdf_path, dpi=300, poppler_path='/opt/homebrew/bin')

# 最初のページを取得
img = imgs[0]

# PIL Image からピクセルサイズを取得
width_px, height_px = img.size


# PDFページの幅・高さ（pt単位, 1pt = 1/72 inch）
cm2inch = 0.393701
page_width_pt = 21 * cm2inch 
page_height_pt = 29.7 * cm2inch

# dpiを計算
width_dpi = width_px / page_width_pt
height_dpi = height_px / page_height_pt

print(f"Pixel size: {width_px} x {height_px}")
print(f"Page size (pt): {page_width_pt:.2f} x {page_height_pt:.2f}")
print(f"Estimated DPI: {width_dpi:.2f} x {height_dpi:.2f}")


Pixel size: 2480 x 3508
Page size (pt): 8.27 x 11.69
Estimated DPI: 299.96 x 300.01
